# Week 44, Day 2: Build a hash map with chaining

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w044d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
buckets = [[] for _ in range(4)]

def put(key, value):
    chain = buckets[key % 4]
    for pair in chain:
        if pair[0] == key:
            pair[1] = value
            return
    chain.append([key, value])

for k, v in [(5, "a"), (9, "b"), (2, "c"), (5, "d"), (13, "e")]:
    put(k, v)
print(buckets)

In [ ]:
class HashMap:
    def __init__(self, capacity=8):
        self._buckets = [[] for _ in range(capacity)]
        self._size = 0

    def _chain(self, key):
        return self._buckets[hash(key) % len(self._buckets)]

    def put(self, key, value):
        chain = self._chain(key)
        for pair in chain:
            if pair[0] == key:
                pair[1] = value                   # an existing key: update it
                return
        chain.append([key, value])                # a new key
        self._size += 1

    def get(self, key, default=None):
        for k, v in self._chain(key):
            if k == key:
                return v
        return default

    def delete(self, key):
        chain = self._chain(key)
        for i, (k, _) in enumerate(chain):
            if k == key:
                chain.pop(i)                       # order inside a chain doesn't matter
                self._size -= 1
                return
        raise KeyError(key)

m = HashMap()
for word in "the cat sat on the mat".split():
    m.put(word, m.get(word, 0) + 1)
print(m.get("the"), m.get("cat"), m.get("dog"), m._size)
m.delete("cat")
print(m.get("cat", "gone"), m._size)

In [ ]:
class HashMap:
    def __init__(self, capacity=8, max_load=0.75):
        self._buckets = [[] for _ in range(capacity)]
        self._size = 0
        self.max_load = max_load
        self.resizes = 0

    def _index(self, key, buckets):
        return hash(key) % len(buckets)

    def put(self, key, value):
        chain = self._buckets[self._index(key, self._buckets)]
        for pair in chain:
            if pair[0] == key:
                pair[1] = value
                return
        chain.append([key, value])
        self._size += 1
        if self._size / len(self._buckets) > self.max_load:
            self._resize(2 * len(self._buckets))

    def _resize(self, capacity):
        new = [[] for _ in range(capacity)]
        for chain in self._buckets:
            for pair in chain:
                new[self._index(pair[0], new)].append(pair)    # every key moves to its new bucket
        self._buckets = new
        self.resizes += 1

    def longest_chain(self):
        return max(len(c) for c in self._buckets)

m = HashMap()
for i in range(1000):
    m.put(i * 7, i)
print(len(m._buckets), m.resizes, m._size / len(m._buckets), m.longest_chain())

In [ ]:
class BadKey:
    def __init__(self, n):
        self.n = n
    def __eq__(self, other):
        return self.n == other.n
    def __hash__(self):
        return 42                                 # every key collides

import time
for kind in [int, BadKey]:
    start = time.perf_counter()
    d = {}
    for i in range(3000):
        d[kind(i)] = i
    print(f"{kind.__name__:<7} {1000 * (time.perf_counter() - start):6.1f} ms")

> **Key idea.** A chaining hash map is an array of buckets, each a list of key-value pairs; a key's bucket is `hash(key) % m`, and the right pair is found by comparing keys with `==`. `put` updates an existing key or appends a new pair; `delete` removes the pair or raises `KeyError`. The load factor α = n/m is the average chain length; when it passes a threshold, double the table and rehash every key (O(1) amortised). Operations are O(1) on average with a good hash and O(n) in the worst case, when everything collides. The special methods `__getitem__`, `__setitem__`, `__delitem__`, `__contains__`, `__len__` and `__iter__` make it work like a dict, though its key order is arbitrary.

### Exercise: A chaining hash map

Write `HashMap(capacity=8)`: `capacity` empty buckets (each a list), with `put(key, value)` (updating an existing key), `get(key, default=None)`, `contains(key)` and `__len__`. Use `hash(key) % len(self._buckets)` to choose a bucket. No resizing yet, and don't use a dict or set inside.

In [ ]:
class HashMap:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_map():\n    \"a map\"\n    m = HashMap(4)\n    for k, v in [(\"a\", 1), (\"b\", 2), (\"c\", 3), (\"a\", 10)]:\n        m.put(k, v)\n    assert len(m) == 3 and m.get(\"a\") == 10 and m.get(\"b\") == 2 and m.get(\"z\") is None and m.get(\"z\", 0) == 0\n    assert m.contains(\"c\") and not m.contains(\"d\") and len(m._buckets) == 4\n    for i in range(100):\n        m.put(i, i * i)\n    assert len(m) == 103 and m.get(9) == 81 and all(isinstance(b, list) for b in m._buckets)\n    m.put(1.0, \"one\")\n    assert m.get(1) == \"one\" and len(m) == 103, \"1 and 1.0 are the same key\"\ndef test_no_dict():\n    \"built from buckets\"\n    import ast\n    assert not any(isinstance(n, (ast.Dict, ast.Set, ast.DictComp, ast.SetComp)) or (isinstance(n, ast.Name) and n.id in (\"dict\", \"set\")) for n in ast.walk(ast.parse(source)))")

### Exercise: Deleting

Add `delete(key)` to the starter's map: remove the key's pair and return its value, or raise `KeyError(key)` if it isn't there. The size must stay right.

In [ ]:
class HashMap:
    def __init__(self, capacity=8):
        self._buckets = [[] for _ in range(capacity)]
        self._size = 0

    def _chain(self, key):
        return self._buckets[hash(key) % len(self._buckets)]

    def put(self, key, value):
        chain = self._chain(key)
        for pair in chain:
            if pair[0] == key:
                pair[1] = value
                return
        chain.append([key, value])
        self._size += 1

    def get(self, key, default=None):
        for k, v in self._chain(key):
            if k == key:
                return v
        return default

    def __len__(self):
        return self._size

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_delete():\n    \"deleting\"\n    m = HashMap(2)\n    for i in range(10):\n        m.put(i, str(i))\n    assert m.delete(4) == \"4\" and m.get(4) is None and len(m) == 9 and m.get(6) == \"6\"\n    try:\n        m.delete(4)\n        assert False\n    except KeyError as err:\n        assert err.args == (4,)\n    for i in range(10):\n        if i != 4:\n            m.delete(i)\n    assert len(m) == 0 and all(b == [] for b in m._buckets)")

### Exercise: Growing the table

Give the starter's map a load-factor limit: after `put` adds a **new** key, if `len / number of buckets` is more than `max_load` (default 0.75), double the number of buckets and rehash every pair into the new ones. Keep a count `resizes` and a read-only property `capacity` (the number of buckets).

In [ ]:
class HashMap:
    def __init__(self, capacity=8, max_load=0.75):
        self._buckets = [[] for _ in range(capacity)]
        self._size = 0
        self.max_load = max_load
        self.resizes = 0

    def _chain(self, key):
        return self._buckets[hash(key) % len(self._buckets)]

    def get(self, key, default=None):
        for k, v in self._chain(key):
            if k == key:
                return v
        return default

    def __len__(self):
        return self._size

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_resize():\n    \"resizing\"\n    m = HashMap(4)\n    caps = []\n    for i in range(13):\n        m.put(f\"k{i}\", i)\n        caps.append(m.capacity)\n    assert caps == [4, 4, 4, 8, 8, 8, 16, 16, 16, 16, 16, 16, 32], caps\n    assert all(m.get(f\"k{i}\") == i for i in range(13)) and len(m) == 13 and m.resizes == 3\n    m.put(\"k0\", \"updated\")\n    assert m.capacity == 32 and m.get(\"k0\") == \"updated\" and m.resizes == 3, \"updating doesn't resize\"\n    try:\n        m.capacity = 3\n        assert False\n    except AttributeError:\n        pass\ndef test_many():\n    \"many keys stay quick\"\n    m = HashMap()\n    for i in range(50_000):\n        m.put(i, i)\n    assert m.capacity == 131072 and max(len(b) for b in m._buckets) <= 3")

### Exercise: Behaving like a dict

Give the starter's map the mapping methods: `m[key]` (raising `KeyError(key)` if missing), `m[key] = value`, `del m[key]`, `key in m`, `len(m)`, iteration over the keys (in any order, without `yield`), `items()` (a list of `(key, value)` tuples), and `__repr__` like `HashMap({'a': 1})` with the pairs **sorted by key**, so it doesn't depend on hashing.

In [ ]:
class HashMap:
    def __init__(self, capacity=8):
        self._buckets = [[] for _ in range(capacity)]
        self._size = 0

    def _chain(self, key):
        return self._buckets[hash(key) % len(self._buckets)]

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_protocol():\n    \"like a dict\"\n    m = HashMap()\n    for w in \"the cat sat on the mat the end\".split():\n        if w in m:\n            m[w] += 1\n        else:\n            m[w] = 1\n    assert m[\"the\"] == 3 and len(m) == 6 and sorted(m) == [\"cat\", \"end\", \"mat\", \"on\", \"sat\", \"the\"]\n    assert sorted(m.items()) == [(\"cat\", 1), (\"end\", 1), (\"mat\", 1), (\"on\", 1), (\"sat\", 1), (\"the\", 3)]\n    del m[\"the\"]\n    assert \"the\" not in m and len(m) == 5 and repr(HashMap()) == \"HashMap({})\"\n    for bad in [lambda: m[\"zebra\"], lambda: m.__delitem__(\"zebra\")]:\n        try:\n            bad()\n            assert False\n        except KeyError:\n            pass\n    small = HashMap()\n    small[\"b\"], small[\"a\"] = 2, 1\n    assert repr(small) == \"HashMap({'a': 1, 'b': 2})\" and \"yield\" not in source")

### Exercise: Measuring the chains

Write `chain_stats(keys, m, hash_fn)`: put the keys (assumed distinct) into `m` buckets by `hash_fn(key) % m`, and return `(longest, average_nonempty, empty)`: the longest chain, the average length of the **non-empty** chains (rounded to 2 places), and how many buckets are empty. Then set `good` and `awful` to the stats for the numbers 0 to 999 in 100 buckets, hashing with `lambda n: n * 2654435761 % 2**32` and with `lambda n: n * 100`.

In [ ]:
def chain_stats(keys, m, hash_fn):
    ___

good = ___
awful = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stats():\n    \"chains\"\n    assert chain_stats([1, 2, 3, 5], 4, lambda n: n) == (2, 1.33, 1) and chain_stats([], 3, lambda n: n) == (0, 0, 3)\n    assert awful == (1000, 1000.0, 99) and good[0] < 25 and good[2] < 5")

### Exercise: A set from a map

A set is a map with no values. Write `HashSet(items=())` on the starter's `HashMap` (composition, Week 30: hold one, don't inherit), with `add`, `discard` (no error if missing), `__contains__`, `__len__`, `__iter__`, and `union(other)` and `intersection(other)` returning new `HashSet`s.

In [ ]:
class HashMap:
    def __init__(self):
        self._buckets = [[] for _ in range(64)]
        self._size = 0

    def _chain(self, key):
        return self._buckets[hash(key) % len(self._buckets)]

    def put(self, key, value):
        chain = self._chain(key)
        for pair in chain:
            if pair[0] == key:
                pair[1] = value
                return
        chain.append([key, value])
        self._size += 1

    def contains(self, key):
        return any(k == key for k, _ in self._chain(key))

    def delete(self, key):
        chain = self._chain(key)
        for i, (k, _) in enumerate(chain):
            if k == key:
                chain.pop(i)
                self._size -= 1
                return
        raise KeyError(key)

    def keys(self):
        return [k for chain in self._buckets for k, _ in chain]

    def __len__(self):
        return self._size

class HashSet:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_set():\n    \"a set\"\n    s = HashSet([3, 1, 3, 2])\n    assert len(s) == 3 and 3 in s and 9 not in s and sorted(s) == [1, 2, 3]\n    s.add(9)\n    s.discard(1)\n    s.discard(100)\n    assert sorted(s) == [2, 3, 9]\n    t = HashSet([3, 4])\n    assert sorted(s.union(t)) == [2, 3, 4, 9] and sorted(s.intersection(t)) == [3] and isinstance(s.union(t), HashSet)\n    assert sorted(s) == [2, 3, 9] and not issubclass(HashSet, HashMap)")

### Bug hunt: The same key twice

`put` always appends a new pair, even when the key is already there, so the map holds two pairs for `"a"`, `len` says 3, and `get` finds the old value: this prints `3 1`. Make `put` update an existing key, so it prints `2 5`.

In [ ]:
class HashMap:
    def __init__(self):
        self._buckets = [[] for _ in range(8)]
        self._size = 0

    def put(self, key, value):
        self._buckets[hash(key) % 8].append([key, value])
        self._size += 1

    def get(self, key):
        for k, v in self._buckets[hash(key) % 8]:
            if k == key:
                return v

m = HashMap()
m.put("a", 1)
m.put("b", 2)
m.put("a", 5)
print(m._size, m.get("a"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"update, don't duplicate\"\n    expect_output(\"2 5\")")

### Bug hunt: Moved, but not rehashed

When this map grows, it copies each bucket into the **same index** of the bigger table instead of rehashing each key, so after a resize most keys are in the wrong bucket and can't be found: this prints a number below 20. Rehash every pair into its new bucket, so it prints `20`.

In [ ]:
class HashMap:
    def __init__(self):
        self._buckets = [[] for _ in range(4)]
        self._size = 0

    def put(self, key, value):
        self._buckets[key % len(self._buckets)].append([key, value])
        self._size += 1
        if self._size > len(self._buckets):
            new = [[] for _ in range(2 * len(self._buckets))]
            for i, chain in enumerate(self._buckets):
                new[i] = chain
            self._buckets = new

    def get(self, key):
        for k, v in self._buckets[key % len(self._buckets)]:
            if k == key:
                return v

m = HashMap()
for k in range(20):
    m.put(k, k)
print(sum(1 for k in range(20) if m.get(k) == k))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"rehash on resize\"\n    expect_output(\"20\")")

### Bug hunt: A fixed number in the formula

This map grows its table, but `_index` still takes the remainder by the **original** 8, so only the first 8 buckets are ever used and the extra space is wasted: this prints `8 56` (buckets used, buckets empty). Use the current number of buckets, so it prints `40 24`.

In [ ]:
class HashMap:
    def __init__(self):
        self._buckets = [[] for _ in range(8)]
        self._size = 0

    def _index(self, key):
        return hash(key) % 8

    def put(self, key, value):
        self._buckets[self._index(key)].append([key, value])
        self._size += 1
        if self._size > len(self._buckets) * 0.75:
            old = self._buckets
            self._buckets = [[] for _ in range(2 * len(old))]
            for chain in old:
                for pair in chain:
                    self._buckets[self._index(pair[0])].append(pair)

m = HashMap()
for k in range(40):
    m.put(k, k)
used = sum(1 for b in m._buckets if b)
print(used, len(m._buckets) - used)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"use the current size\"\n    expect_output(\"40 24\")")

### Puzzle: Get from a chain

Arrange the lines into a hash map's `__getitem__`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Answer without running code.

p = ___   # the load factor of a map with 12 keys in 16 buckets
q = ___   # the bucket for key 21 in a table of 8 buckets, using hash(n) == n
r = ___   # after doubling to 16 buckets, the bucket for key 21
s = ___   # the average-case cost of get with a good hash and a bounded load factor: "O(1)" or "O(n)"
t = ___   # the worst-case cost of get if every key has the same hash: "O(1)" or "O(n)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 0.75 and q == 5 and r == 5, \"p, q and r\"\n    assert s == \"O(1)\" and t == \"O(n)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a fuller map

Write `HashMap` with chaining that grows **and shrinks**: it starts with 8 buckets; after adding a new key, if the load factor is above 0.75, double; after removing a key, if it's below 0.25 and there are more than 8 buckets, halve (rehashing every key either way). Give it all of:

- `m[key]`, `m[key] = value`, `del m[key]`, `in`, `len`, iteration over keys, and a read-only `capacity`;
- `get(key, default=None)`, `pop(key, *default)` (like `dict.pop`: removes and returns the value; if missing, returns the default if one was given, else raises `KeyError`), `setdefault(key, default=None)` (returns the value, first storing `default` if the key is missing);
- `update(other)` (from a dict or another `HashMap`), and `__eq__`: equal to another `HashMap` **or a dict** with the same keys and values.

In [ ]:
class HashMap:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_basics():\n    \"the basics\"\n    m = HashMap()\n    m[\"a\"] = 1\n    m.update({\"b\": 2, \"c\": 3})\n    assert m == {\"a\": 1, \"b\": 2, \"c\": 3} and len(m) == 3 and sorted(m) == [\"a\", \"b\", \"c\"]\n    assert m.setdefault(\"a\", 9) == 1 and m.setdefault(\"z\", 26) == 26 and m[\"z\"] == 26\n    assert m.pop(\"z\") == 26 and m.pop(\"z\", \"none\") == \"none\" and m.get(\"q\") is None\n    try:\n        m.pop(\"z\")\n        assert False\n    except KeyError:\n        pass\n    n = HashMap()\n    n.update(m)\n    assert n == m and m == n and (m == [1]) is False\n    del n[\"a\"]\n    assert n != m\ndef test_sizes():\n    \"growing and shrinking\"\n    m = HashMap()\n    for i in range(100):\n        m[i] = i\n    assert m.capacity == 256\n    for i in range(95):\n        del m[i]\n    assert m.capacity == 16 and sorted(m) == [95, 96, 97, 98, 99]\n    for i in range(95, 100):\n        del m[i]\n    assert m.capacity == 8 and len(m) == 0\ndef test_random():\n    \"against a dict\"\n    rng = random.Random(5)\n    m, ref = HashMap(), {}\n    for _ in range(5000):\n        k = rng.randint(0, 300)\n        op = rng.random()\n        if op < 0.5:\n            m[k] = ref[k] = rng.randint(0, 9)\n        elif op < 0.8 and k in ref:\n            del m[k], ref[k]\n        else:\n            assert m.get(k, \"x\") == ref.get(k, \"x\")\n    assert m == ref and len(m) == len(ref)")

## Warm-up

Keys 5, 9 and 13 all have remainder 1, so they share slot 1's chain; putting 5 a second time **updated** its pair instead of adding another. That's a whole hash map, in miniature: the rest of today is making it correct, fast and convenient.

## Chains of pairs

A chaining hash map is an array of `m` **buckets**, each a list of `[key, value]` pairs:

```text
 index = hash(key) % m

 bucket 0: []
 bucket 1: [[5, "d"], [9, "b"], [13, "e"]]      <- a chain: keys that collided
 bucket 2: [[2, "c"]]
 bucket 3: []
```

Every operation does the same two things: work out the bucket from the key's hash, then search that one **short** chain, comparing keys with `==`. The hash narrows the search to one bucket; equality picks the right key within it. That's why Week 29 insisted that equal keys must have equal hashes: otherwise two equal keys could sit in different buckets, and neither would ever be found by the other.

(Chains can be linked lists of nodes, as in Week 42, and in many languages they are. In Python, a short list of pairs is simpler and faster.)

## Getting, putting, deleting

Three details are easy to miss: `put` must **update** an existing key rather than append a second pair (or `len` and lookups go wrong); `delete` must raise `KeyError` for a missing key, like a dict; and the size counter must change only when a key is really added or removed.

## The load factor and resizing

The **load factor** is the number of keys divided by the number of buckets: α = n / m. With a good hash, an average chain holds α keys, so a lookup checks about α pairs. Keep α small and lookups stay fast; let it grow and the map slowly turns into a handful of long lists.

So when α passes a threshold (0.75 is a common choice), **resize**: make a table about twice as big and **rehash** every key into it. Every key must be re-placed, because its bucket depends on `m`: a key in bucket 5 of 8 might belong in bucket 13 of 16.

A resize is O(n), but like Week 41's dynamic array, doubling makes it rare enough that `put` is O(1) **amortised**: n puts cause resizes copying fewer than 2n keys in total.

## How fast, really?

| | Average (good hash, α kept below a constant) | Worst case (every key in one bucket) |
|---|---|---|
| get, put, delete | O(1) | O(n) |
| resize | O(n), but O(1) amortised per put | O(n) |
| iterate over all keys | O(n + m) | O(n + m) |

The average depends on the keys being spread evenly, which depends on the hash function (Day 1). The worst case is real: a bad hash function, or an attacker who knows the hash (hash flooding), puts everything in one chain. That's why Python randomises its string hashes, and why some languages switch long chains to balanced trees.

Even Python's own, carefully built dictionary slows to a crawl when every key has the same hash.

## Behaving like a dict

Week 31's container protocols let a hash map work with Python's own syntax: `__getitem__` for `m[key]` (raising `KeyError` if missing), `__setitem__` for `m[key] = value`, `__delitem__` for `del m[key]`, `__contains__` for `in`, `__len__` and `__iter__` (over the keys, as a dict does). With those, `m["cat"] += 1` and `for key in m` work as expected.

One difference from a real dict: our map's keys come out in **bucket order**, which depends on the hashes, so it's effectively arbitrary (and, with string keys, different each run). Python's dict remembers the order keys were added; Day 4 shows how.

## Practice

## Quiz

## Daily challenge

## Before you go

Explain to yourself, without notes, what goes wrong if `put` forgets to check for an existing key, and what goes wrong if a resize copies buckets without rehashing. Tomorrow: open addressing, where there are no chains at all and every key lives directly in the table.